# ApexPitwall: Exploratory Data Analysis of Tyre Degradation
This notebook evaluates lap times vs tyre age across dry compounds (Soft, Medium, Hard) and circuits for the 2022 season.
We also investigate key confounders: fuel mass burn and track temperature.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/processed/race_laps_processed.csv')
print(f'Total processed laps: {len(df)}')
df.head()

## 1. Clean Laps Summary per Circuit and Compound

In [ ]:
summary = df.groupby(['Circuit', 'Compound']).agg(
    LapCount=('lap_time_seconds', 'count'),
    MeanLapTime=('lap_time_seconds', 'mean'),
    StdLapTime=('lap_time_seconds', 'std'),
    MinTyreLife=('TyreLife', 'min'),
    MaxTyreLife=('TyreLife', 'max'),
    MeanTrackTemp=('TrackTemp', 'mean')
).round(3)
summary

## 2. Unadjusted Tyre Degradation Slopes
Notice how without controlling for fuel mass burn, later stints on lighter cars exhibit negative unadjusted slopes.

In [ ]:
slopes = []
for (circuit, compound), group in df.groupby(['Circuit', 'Compound']):
    if len(group) >= 15:
        p = np.polyfit(group['TyreLife'], group['lap_time_seconds'], 1)
        slopes.append({'Circuit': circuit, 'Compound': compound, 'Laps': len(group), 'Slope_s_per_lap': round(p[0], 4), 'BasePace_s': round(p[1], 3)})
pd.DataFrame(slopes)